Validate every order: good ones are counted, bad ones go to the dead-letter topic.

In [3]:
from confluent_kafka import Producer, Consumer

In [4]:
%run order.ipynb

In [5]:
consumer = Consumer({"bootstrap.servers": "localhost:9092",
                     "group.id": "order-validator",
                     "auto.offset.reset": "earliest",
                     "enable.auto.commit": False,
                     "session.timeout.ms": 6000})
producer = Producer({"bootstrap.servers": "localhost:9092",
                     "acks": "all"})
consumer.subscribe(["orders"])

In [6]:
def to_dlq(msg, reason):
    headers = {"error": reason, "source": f"{msg.topic()}/{msg.partition()}/{msg.offset()}"}
    producer.produce("orders-dlq", key=msg.key(), value=msg.value(), headers=headers)
    print(f"to DLG: partition {msg.partition()} offset {msg.offset()}: {reason}")
        

In [7]:
good, bad, idle = 0, 0, 0

while idle < 5:
    msg= consumer.poll(1.0)
    if msg is None:
        if consumer.assignment():
            idle += 1
        continue
    if msg.error():
        print("error", msg.error())
        continue
    idle = 0

    try:
        order = decode(msg.value())
    except ValueError as e:
        to_dlq(msg, f"not json {e}")
        bad += 1
        continue
    problems = validate(order)
    if problems:
        to_dlq(msg, ", ".join(problems))
        bad += 1
    else:
        good += 1

producer.flush()
if good + bad:
    consumer.commit(asynchronous=False)
consumer.close()
print(f"valid orders: {good}, sent to the dead-letter topic: {bad}")

to DLG: partition 0 offset 25: not json Expecting value: line 1 column 1 (char 0)
to DLG: partition 0 offset 26: missing field city
to DLG: partition 0 offset 27: qty must be at least 1
valid orders: 54, sent to the dead-letter topic: 3


In [ ]:
# To check
# kafka-console-consumer.sh --bootstrap-server $BS --topic orders-dlq --from-beginning --max-messages 3 --formatter-property print.headers=true
"""
The consumer rebalance protocol (KIP-848) is production-ready! Set group.protocol=consumer to try it out. See https://kafka.apache.org/documentation/#consumer_rebalance_protocol
error:not json Expecting value: line 1 column 1 (char 0),source:orders/0/25     this is not json
error:missing field city,source:orders/0/26     {"schema_version": 1, "order_id": "o-09001", "customer_id": "c-010", "product_id": "P300", "qty": 2, "price": 15.0, "ts": "2026-11-21T02:20:30"}
error:qty must be at least 1,source:orders/0/27 {"schema_version": 1, "order_id": "o-9002", "customer_id": "c-010", "product_id": "P300", "qty": 0, "price": 15.0, "city": "Manila", "ts": "2026-11-21T02:20:30"}
Processed a total of 3 messages
"""